##Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Data Cleaning
Reading in the dataframe containing the player names for each of the top 20 teams:

In [ ]:
#df_rosters = pd.read_csv('drive/MyDrive/Colab Notebooks/Data Project/dataframes/df_rosters.csv')
df_rosters = pd.read_csv('drive/MyDrive/Colab Notebooks/Project Documents/Copy of df_rosters.csv')
df_rosters.head()

,Unnamed: 0,Team,Player,First,Last
0,0,North Carolina (Darkside),Thomas Harley,Thomas,Harley
1,1,North Carolina (Darkside),Alex Fiordalisi,Alex,Fiordalisi
2,2,North Carolina (Darkside),Elijah Danley,Elijah,Danley
3,3,North Carolina (Darkside),Aaron Wei,Aaron,Wei
4,4,North Carolina (Darkside),Samuel Redinbo,Samuel,Redinbo


Reading in the dataframe containing top 20 teams of the college division, ranked in order:

In [ ]:
#df_rankings = pd.read_csv('drive/MyDrive/Colab Notebooks/Data Project/dataframes/team_rankings.csv')
df_rankings = pd.read_csv('drive/MyDrive/Colab Notebooks/Project Documents/Copy of team_rankings.csv')

#Updating the 20th ranked team, as Ultiworld mistakenly used a D-III team, and we are focussed on D-I
df_rankings.iloc[19] = ['20', 'North Carolina State',	'1638',	'College',	'Men',	'Division I',	'Atlantic Coast',	'Carolina DI', '20',	'18']

#Cleaning the dataframe to only include the columns we want
df_rankings = df_rankings.drop(["Competition Level", "Gender Division", "Competition Division", "Wins", "Losses"], axis = 1)
df_rankings.head()

,Rank,Team,Power Rating,College Region,College Conference
0,1,North Carolina,2124,Atlantic Coast,Carolina DI
1,2,Massachusetts,2058,New England,Greater New England DI
2,3,Cal Poly-SLO,1992,Southwest,SoCal DI
3,4,Colorado,1969,South Central,Rocky Mountain DI
4,5,Vermont,1940,New England,Greater New England DI


Reading in the csv as a dataframe containing every article written about the d1 Men's college division in 2023:

In [ ]:
#df_articles = pd.read_csv('drive/MyDrive/Colab Notebooks/Data Project/dataframes/Newest Copy of articles.csv')
df_articles = pd.read_csv('drive/MyDrive/Colab Notebooks/Project Documents/Newest Copy of articles.csv')
df_articles.head()

,author,text,date,title
0,EDWARD STEPHENS,Recognizing the top seven performers of the 20...,"June 12, 2023",2023 D-I Men’s All-American First Team
1,EDWARD STEPHENS,Recognizing the top performer of the 2023 seas...,"June 8, 2023",2023 D-I Men’s Player of the Year: Pittsburgh’...
2,None,The winners of our College Championships games...,"June 7, 2023",College Championships 2023: D-I and D-III #The...
3,None,"Box scores, including yardage data, from the f...","June 1, 2023",D-I College Championships 2023: Final Box Scores
4,ALEX RUBIN,"Who won the twitterverse?\nMAY 31, 2023 BY ALE...","May 31, 2023",Best Tweets of the 2023 D-I College Championships


Cleaning the articles so that they are formatted correctly, and removed any names that will cause a duplication of occurance results:

In [ ]:
#df_articles["text"] = df_articles["text"]
df_articles["text"] = df_articles["text"].str.replace("\n", "")

# List of teams to remove since they are similar to ones we want
removable_words = ["UMass-Lowell", 'Carleton University', "Texas A&M", "Texas State", "Texas-San Antonio", "North Texas", "South Texas"
,"CaliforniaState", "Southern California", "NorCal", "Washington State", "Western Washington", "George Washington", "Minnesota Duluth", "UNCC"]

pattern = '|'.join(rf'\b{word}\b' for word in removable_words)
df_articles['text'] = df_articles['text'].str.replace(pattern, '', case=False)

<ipython-input-220-f235f0b587d6>:9: FutureWarning:

The default value of regex will change from True to False in a future version.



Consolidating the different player names for each team into one dataframe:

In [ ]:
team_names = {}

def get_team_name(x):
  names = x.split('(')
  return names[1].replace(')', '')

def f_the_paren(x):
  return x.split('(')[0]

df_rosters['Team Name'] = df_rosters['Team'].map(get_team_name)
df_rosters['Team'] = df_rosters['Team'].map(f_the_paren)

#Dictionary Creation
Creating a dictionary to hold the team names, and any possible nicknames

In [ ]:
team_names = {}
for team, team_name in zip(df_rosters['Team'], df_rosters["Team Name"]):
  team = team.strip(' ')
  team_name = team_name.strip(' ')

  if team not in team_names:
    team_names[team] = [team_name]

for team in df_rosters['Team']:
  team = team.strip(' ')
  if team not in team_names[team]:
    team_names[team].append(team)

#Adding each player's FULL names to the dictionary per team
for team, player in zip(df_rosters['Team'], df_rosters["Player"]):
  team = team.strip(' ')
  team_name = team_name.strip(' ')
  team_names[team].append(player)

Cleaning the dictionary to include all possible nicknames so we can search for appearances. Some names were manipulated to remove any possible duplication when searching for count.

**Team Mentions**

In [ ]:
df_mentions = pd.DataFrame()
df_mentions['Team'] = team_names.keys()
team_names["North Carolina"].append("UNC ")
team_names["North Carolina"].append("UNC,")
team_names["North Carolina"].append("UNC.")
team_names["North Carolina"].append("UNC's")

team_names["Brown"].remove("Brownian Motion")
team_names["Brown"].append("Motion")

team_names["Massachusetts"].append("Mass")
team_names["Massachusetts"].append("Zoo's")
team_names["Massachusetts"].append("Zoo ")
team_names["Massachusetts"].append("ZooDisc")
team_names["Massachusetts"].remove("Massachusetts")

team_names["Cal Poly-SLO"].append("SLO")
team_names["Cal Poly-SLO"].remove("Cal Poly-SLO")
team_names["Cal Poly-SLO"].remove("SLOCORE")
team_names["Cal Poly-SLO"].append("CalPoly")

team_names["California-Santa Cruz"].remove("California-Santa Cruz")

team_names["California"].append("Cal,")
team_names["California"].append("Cal ")
team_names["California"].append("Cal.")
team_names["California"].append("(Cal)")

team_names["California-Santa Cruz"].append("UCSC")
team_names["California-Santa Cruz"].append("Santa Cruz")

team_names["Carleton College"].append("Carleton")
team_names["Carleton College"].remove("Carleton College")

team_names["North Carolina State"].append("NC State")
team_names["North Carolina State"].append("Alpha")
team_names["North Carolina State"].remove("NC State Alpha")

team_names["Pittsburgh"].append("Pitt")
team_names["Pittsburgh"].remove("Pittsburgh")

team_names["British Columbia"].append("UBC")

##Data Descriptions
Gathering the total number of articles each team is mentioned

In [ ]:
mentions = [0] * 20

for article in df_articles["text"]:
  count = 0
  for key in team_names:
    for team_name in team_names[key]:
      if team_name in article:
         mentioned = 1
    mentions[count] += mentioned
    mentioned = 0
    count += 1

df_mentions["total articles mentioned"] = mentions
df_mentions.head()

,Team,total articles mentioned
0,North Carolina,39
1,Massachusetts,36
2,Cal Poly-SLO,36
3,Colorado,38
4,Vermont,38


Searching through each article for the total number of occurances per team

In [ ]:
team_occurrences = {team: [] for team in team_names}

for index, row in df_articles.iterrows():
    article_text = row['text']
    article_counts = {}

    for team, possible_names in team_names.items():
        count = sum(article_text.count(name) for name in possible_names)
        article_counts[team] = count
        team_occurrences[team].append(count)

    for team, count_list in article_counts.items():
        df_articles.at[index, team] = count_list

Taking the sum of each team and putting it into the teams dataframe

In [ ]:
team_columns = ['North Carolina', 'Massachusetts', 'Cal Poly-SLO', 'Colorado', 'Vermont', "Oregon", "Brown", "Pittsburgh",
                "California-Santa Cruz", "Texas", "Carleton College", "Minnesota", "UCLA", "Georgia","British Columbia", "California", "Tufts", "Washington", "Utah State", "North Carolina State"]

# Create a new DataFrame with the sums of occurrences for each team
team_sums_df = pd.DataFrame(df_articles[team_columns].sum(axis=0), columns=['Total Mentions']).reset_index()
team_sums_df.columns = ['Team', 'Total Mentions']
team_sums_df = team_sums_df.merge(df_rankings, how = "outer")

#Removes all articles with ZERO mentions of any of the wanted teams, so there are only 97 articles left
df_articles["All mentions"] = df_articles.iloc[:,-20:].sum(axis=1)
df_articles = df_articles[df_articles['All mentions'] != 0]
team_sums_df.head()

,Team,Total Mentions,Rank,Power Rating,College Region,College Conference
0,North Carolina,510.0,1,2124,Atlantic Coast,Carolina DI
1,Massachusetts,324.0,2,2058,New England,Greater New England DI
2,Cal Poly-SLO,363.0,3,1992,Southwest,SoCal DI
3,Colorado,232.0,4,1969,South Central,Rocky Mountain DI
4,Vermont,222.0,5,1940,New England,Greater New England DI


Looking at the total mention descripitive data, we found for all of the teams there was a mean of 190 mentions, a median a 164 mentions, and a standard deviation of 112 mentions. The data ranges from 510 to 42 total mentions across all 20 teams.

In [ ]:
team_sums_df["Total Mentions"].mean(), team_sums_df["Total Mentions"].std(), team_sums_df["Total Mentions"].median()

(189.4, 111.19750091852927, 163.5)

In [ ]:
df_articles

,author,text,date,title,North Carolina,Massachusetts,Cal Poly-SLO,Colorado,Vermont,Oregon,...,Minnesota,UCLA,Georgia,British Columbia,California,Tufts,Washington,Utah State,North Carolina State,All mentions
0,EDWARD STEPHENS,Recognizing the top seven performers of the 20...,"June 12, 2023",2023 D-I Men’s All-American First Team,19.0,12.0,2.0,2.0,1.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,61.0
1,EDWARD STEPHENS,Recognizing the top performer of the 2023 seas...,"June 8, 2023",2023 D-I Men’s Player of the Year: Pittsburgh’...,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0
2,None,The winners of our College Championships games...,"June 7, 2023",College Championships 2023: D-I and D-III #The...,10.0,4.0,12.0,5.0,2.0,0.0,...,1.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,51.0
3,None,"Box scores, including yardage data, from the f...","June 1, 2023",D-I College Championships 2023: Final Box Scores,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
4,ALEX RUBIN,"Who won the twitterverse?MAY 31, 2023 BY ALEX ...","May 31, 2023",Best Tweets of the 2023 D-I College Championships,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,14.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,THEO WAN,The USA College Season is Officially here!JANU...,"January 26, 2023",Huckin’ Eh: Bellingham Recap & Santa Barbara P...,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,3.0
158,EDWARD STEPHENS,We preview teams #1-5 in our Power Rankings he...,"January 20, 2023",2023 D-I College Preseason Power Rankings: #1-5,10.0,0.0,0.0,4.0,3.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,34.0
159,EDWARD STEPHENS,We preview teams #6-15 in our Power Rankings h...,"January 19, 2023",2023 D-I College Preseason Power Rankings: #6-15,0.0,4.0,15.0,0.0,0.0,0.0,...,5.0,6.0,5.0,0.0,0.0,0.0,5.0,0.0,6.0,63.0
160,EDWARD STEPHENS,We preview teams #16-25 in our Power Rankings ...,"January 18, 2023",2023 D-I College Preseason Power Rankings: #16-25,0.0,0.0,0.0,0.0,0.0,13.0,...,0.0,0.0,0.0,0.0,5.0,5.0,1.0,4.0,0.0,30.0


Comparing linear regression models

In [ ]:
#Using Rank as a test
X_train = team_sums_df[["Rank"]]
X_test = team_sums_df[["Rank"]]
y_train = team_sums_df["Total Mentions"]

model = LinearRegression()
model.fit(X=X_train, y=y_train)
model.predict(X=X_test)

X_new = pd.DataFrame()
X_new["Rank"] = np.linspace(0, 20, num=20)

y_new_ = pd.Series(
    model.predict(X_new),
    index=X_new["Rank"])

In [ ]:
#Using power rating as the test
power_train = team_sums_df[["Power Rating"]]
power_test = team_sums_df[["Power Rating"]]

model = LinearRegression()
model.fit(X=power_train, y=y_train)
model.predict(X=power_test)

X_new = pd.DataFrame()
X_new["Power Rating"] = np.linspace(1500, 2200, num=20)

y_power_ = pd.Series(
    model.predict(X_new),
    index=X_new["Power Rating"])

Comparing both tests using RMSE
Finding the RMSE for both linear regression tests. We found that when using the Power Rating for the linear regression, our model's predictions are only off by around 52 total mentions, where when using Rank the model's prediction is off around 61 total mentions. These are both good, since the total mentions range from 510 to 42 with a standard deviation of 111 total mentions.

In [ ]:
pipeline = make_pipeline(
          LinearRegression())

#Tetsing the Rank versus total mentions
pipeline.fit(X=X_train, y=y_train)
y_train_ = pipeline.predict(X=X_train)
mseRank = ((y_train - y_train_) ** 2).mean()
rmseRank = np.sqrt(mseRank)

#Testing the Power Rating versus total mentions
pipeline.fit(X=power_train, y=y_train)
y_train_ = pipeline.predict(X=power_train)
msePower = ((y_train - y_train_) ** 2).mean()
rmsePower = np.sqrt(msePower)

rmseRank, rmsePower, team_sums_df["Total Mentions"].std()

(60.52039425103535, 51.6153897772586, 111.19750091852927)

Linear Regression Plot (found using the Python directory)
- While this doesn't use our found linear regression model, it runs its own linear regression and plots it using trendline = "ols"

In [ ]:
#Need to cast for the linear regression "ols"
team_sums_df["Rank"] = team_sums_df["Rank"].astype(float)
team_sums_df["Power Rating"] = team_sums_df["Power Rating"].astype(float)

colors = ['black'] * len(team_sums_df)
calPoly = 2
colors[calPoly] = "green"

fig = px.scatter(team_sums_df, x="Power Rating", y="Total Mentions", color = colors, hover_data=['Team'], trendline = "ols",
                 trendline_color_override = '#194c62', title = "Linear Regression for Power Rating vs Total Mentions")
fig.update_xaxes(tickvals=[1600, 1700, 1800, 1900, 2000, 2100, 2200])

#Adding labels for certain points
fig.add_annotation(
        x=1992, y=363, xref="x", yref="y", text="Cal Poly", showarrow=True,
        font=dict(family="Courier New, monospace", size=16, color="#ffffff"),
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="#636363", ax=80, ay=-30, bordercolor="#194c62", borderwidth=2, borderpad=4, bgcolor="#3f8aa9",opacity=0.8)
fig.add_annotation(x=2124, y=510, xref="x", yref="y", text="North Carolina", showarrow=True,
        font=dict( family="Courier New, monospace", size=16, color="#ffffff"
            ), align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="#636363", ax=80, ay=-30, bordercolor="grey", borderwidth=2, borderpad=4, bgcolor="grey",opacity=0.8)
fig.add_annotation(x=1880, y=133, xref="x", yref="y", text="UC Santa Cruz", showarrow=True,
        font=dict(
            family="Courier New, monospace", size=16, color="#ffffff"
            ), align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="#636363", ax=100, ay=-10, bordercolor="grey", borderwidth=2, borderpad=4, bgcolor="grey",opacity=0.8)

fig.update_layout(showlegend=False, width=900, height=700, yaxis_tickfont=dict(size=16), xaxis_title_font=dict(size=20), yaxis_title_font=dict(size=20), xaxis_tickfont=dict(size=16), title_font = dict(family= "Impact, sans-serif", size = 28))
fig.show()

#Analyzing different authors
Since Cal Poly is mentioned more than we predicted given their Rank and Power Rating, we decided to look into factors that would affect their mentions. The first one we look at is the authors.

In [ ]:
#Looking at the total times each author mentions each team
author_sums = df_articles.groupby('author').sum()

#Looking at the proportions each team is mentioned per author
author_proportions = author_sums.div(author_sums.sum(axis=1), axis=0)
author_proportions = author_proportions.reset_index()

<ipython-input-233-db37ae7d8df7>:2: FutureWarning:

The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.



In [ ]:
author_proportions.sort_values("Cal Poly-SLO", ascending = False).head(n=3)

,author,North Carolina,Massachusetts,Cal Poly-SLO,Colorado,Vermont,Oregon,Brown,Pittsburgh,California-Santa Cruz,...,Minnesota,UCLA,Georgia,British Columbia,California,Tufts,Washington,Utah State,North Carolina State,All mentions
15,PATRICK STEGEMOELLER,0.044118,0.250000,0.132353,0.000000,0.000000,0.014706,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.014706,0.000000,0.000000,0.000000,0.5
14,None,0.103774,0.037736,0.113208,0.056604,0.018868,0.000000,0.028302,0.056604,0.009434,...,0.009434,0.000000,0.047170,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.5
10,JAKE THORNE,0.016282,0.010176,0.104478,0.046811,0.010855,0.052917,0.019674,0.011533,0.053596,...,0.009498,0.015604,0.012212,0.007463,0.040706,0.007463,0.025102,0.015604,0.006784,0.5


In [ ]:
author_sums.dropna()
author_sums = author_sums.sort_values("Cal Poly-SLO", ascending = False).head(n=5)

fig = px.bar(author_sums, y='Cal Poly-SLO', title = "Top 5 Authors That Mention Cal Poly", color_discrete_sequence=['#194c62'])
fig.update_layout(yaxis_title='Number of Cal Poly mentions', xaxis_title = "Author", width=900, height=800,
                  yaxis_tickfont=dict(size=16), xaxis_title_font=dict(size=20), yaxis_title_font=dict(size=20), title_font = dict(family= "Impact, sans-serif", size = 28))

fig.show()

This is extremely important to us, since Jake Thorne, who is responsible for just under half of all Cal Pol.y's mentions, is a recent Cal Poly graduate and SLOCORE alumni. After finding this out, we decided to run some prediction models to see how much of an effect the author had on the number of mentions for teams.